# English → Twi Fine-tuning with NLLB-200

Fine-tunes `facebook/nllb-200-distilled-600M` on the project's parallel English-Twi corpus using LoRA, with research-grade evaluation (sacreBLEU + chrF++ + COMET-22) on both a held-out in-domain test set and the externally-citable **FLORES-200 devtest** for `twi_Latn`.

Designed to run on:
- **Apple M1 Pro / 16 GB** (MPS, bf16) — development and short runs.
- **Google Colab T4 / A100** (CUDA, fp16/bf16) — full-data final run.

Device-aware: batch size and grad accumulation auto-adjust based on detected hardware. No bitsandbytes (`paged_adamw_32bit` does not work on MPS).

Companion module: `src/translation_eval.py` provides the evaluation pipeline.

The existing `finetune_twi.ipynb` (Llama-3.2-1B) is intentionally untouched so both notebooks can be compared in the paper.

## 1. Environment, dependencies, device detection

In [ ]:
# Run this cell once per environment. Uncomment the line that matches your platform.
# On Colab you may need to restart the runtime after installing.

# !pip install -q -U "torch>=2.10" "transformers>=5.0" "peft>=0.19" "datasets>=4.0" "accelerate>=1.10" sacrebleu sentencepiece evaluate unbabel-comet tensorboard

In [1]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

# Make `src/` importable when running from notebooks/ or the project root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_utils import load_parallel_corpus, make_splits, subsample_train
from src.translation_eval import (
    compute_chrf_bleu,
    evaluate_translation,
    paired_bootstrap,
    print_sample_table,
)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

/Users/joetib/Desktop/projects/graduation/venv/lib/python3.12/site-packages/sklearn/__init__.py:82: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  import scipy.linalg  # noqa
/Users/joetib/Desktop/projects/graduation/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


PROJECT_ROOT = /Users/joetib/Desktop/projects/graduation


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.bfloat16  # M1 supports bf16 natively
elif torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

print(f"DEVICE={DEVICE}  DTYPE={DTYPE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE=mps  DTYPE=torch.bfloat16


In [3]:
# HF token from environment, never hardcoded. Set HF_TOKEN before running this notebook
# if you want to push to the Hub. Reading the dataset and base model is anonymous.
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import login

    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to HuggingFace Hub.")
else:
    print("HF_TOKEN not set — anonymous access only (push-to-hub disabled).")

HF_TOKEN not set — anonymous access only (push-to-hub disabled).


## 2. Model & tokenizer

In [4]:
MODEL_ID = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
TGT_LANG = "twi_Latn"  # NLLB-200 supports Twi natively
MAX_LENGTH = 128  # >99% of dataset/final.json sentences fit; bump if your data is longer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, dtype=DTYPE)
model.to(DEVICE)
print(model.config.architectures, "loaded with dtype", next(model.parameters()).dtype)

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/commits/main "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/discussions?p=0 "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/commits/refs%2Fpr%2F45 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/refs%2Fpr%2F45/model.safetensors.index.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/refs%2Fpr%2F45/model.safetensors "HTTP/1.1 302 Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8

['M2M100ForConditionalGeneration'] loaded with dtype torch.bfloat16


In [5]:
# Sanity check: confirm the LoRA target_modules below exist in this architecture.
# Prints a sorted set of the leaf module name suffixes — you should see q_proj, k_proj, v_proj, out_proj, fc1, fc2.
seen = set()
for name, _ in model.named_modules():
    leaf = name.split(".")[-1]
    if leaf in {"q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"}:
        seen.add(leaf)
print("Found LoRA target module names:", sorted(seen))

Found LoRA target module names: ['fc1', 'fc2', 'k_proj', 'out_proj', 'q_proj', 'v_proj']


## 3. Data: load, split, tokenize

Loads `dataset/final.json` (~594k parallel pairs), strips stray whitespace, and produces a deterministic 99 / 0.5 / 0.5 train / val / test split (seed 42). Test-set indices are written to `results/test_indices.json` so the held-out set is documented for the paper.

In [12]:
MAX_TRAIN_SAMPLES = int(os.environ.get("MAX_TRAIN_SAMPLES", "0")) or 10000

In [13]:
DATA_PATH = PROJECT_ROOT / "dataset" / "final.json"
raw = load_parallel_corpus(str(DATA_PATH))
splits = make_splits(
    raw, seed=SEED, val_size=3000, test_size=3000,
    test_indices_out=str(PROJECT_ROOT / "results" / "test_indices.json"),
)

# Optional dev cap: set MAX_TRAIN_SAMPLES=200 in your shell for a smoke test.

splits = subsample_train(splits, MAX_TRAIN_SAMPLES, seed=SEED)

print({k: len(v) for k, v in splits.items()})

{'train': 10000, 'validation': 3000, 'test': 3000}


In [14]:
def tokenize_pair(batch):
    return tokenizer(
        text=batch["english"],
        text_target=batch["twi"],
        max_length=MAX_LENGTH,
        truncation=True,
    )


tokenized = splits.map(
    tokenize_pair,
    batched=True,
    remove_columns=["english", "twi"],
    desc="Tokenizing",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

Tokenizing:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3000 [00:00<?, ? examples/s]

## 4. LoRA configuration

Conventional ratio `alpha = 2 * r`. Targets cover both attention (`q/k/v/out_proj`) and FFN (`fc1/fc2`) projections in NLLB's encoder and decoder.

In [15]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
)
model = get_peft_model(model, peft_config)
# Required when combining PEFT with gradient_checkpointing: the base model's
# input embeddings do not carry requires_grad=True, so a checkpointed backward
# pass cannot propagate gradients into the LoRA adapters. This hook fixes that.
model.enable_input_require_grads()
model.print_trainable_parameters()

/Users/joetib/Desktop/projects/graduation/venv/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
trainable params: 8,650,752 || all params: 623,724,544 || trainable%: 1.3870


## 5. Training arguments (device-aware)

On MPS, `bf16=True` in `Seq2SeqTrainingArguments` is unstable; the model is already loaded in bf16, so the Trainer flag is left off on M1. On CUDA, the matching flag is enabled.

In [16]:
OUTPUT_DIR = str(PROJECT_ROOT / "results" / "nllb-twi-lora")

if DEVICE == "mps":
    per_device_train_batch_size = 4
    per_device_eval_batch_size = 8
    gradient_accumulation_steps = 8  # effective batch ~32
    use_bf16 = False  # model is in bf16; avoid Trainer flag on MPS
    use_fp16 = False
elif DEVICE == "cuda":
    per_device_train_batch_size = 16
    per_device_eval_batch_size = 32
    gradient_accumulation_steps = 2  # effective batch ~32
    use_bf16 = DTYPE == torch.bfloat16
    use_fp16 = DTYPE == torch.float16
else:
    per_device_train_batch_size = 2
    per_device_eval_batch_size = 4
    gradient_accumulation_steps = 16
    use_bf16 = False
    use_fp16 = False

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    optim="adamw_torch",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=4,
    report_to=["tensorboard"],
    seed=SEED,
)
print("per_device_train_batch_size:", per_device_train_batch_size)
print("effective batch:", per_device_train_batch_size * gradient_accumulation_steps)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


per_device_train_batch_size: 4
effective batch: 32


## 6. Train

In [17]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda ep: compute_chrf_bleu(ep, tokenizer),
)
# To resume from the most recent checkpoint, call trainer.train(resume_from_checkpoint=True)
trainer.train()

/Users/joetib/Desktop/projects/graduation/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Bleu,Chrf
626,12.683555,1.449937,27.479258,50.702767


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"


TrainOutput(global_step=626, training_loss=13.03724353534345, metrics={'train_runtime': 5246.733, 'train_samples_per_second': 3.812, 'train_steps_per_second': 0.119, 'total_flos': 1545783037132800.0, 'train_loss': 13.03724353534345, 'epoch': 2.0})

In [19]:
BEST_DIR = str(PROJECT_ROOT / "results" / "nllb-twi-lora-best")
trainer.model.save_pretrained(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)
print("Saved LoRA adapter to", BEST_DIR)

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"


Saved LoRA adapter to /Users/joetib/Desktop/projects/graduation/results/nllb-twi-lora-best


## 7. Inference sanity check

Loads the base model + trained LoRA adapter and translates 10 hand-picked English sentences. If the output is gibberish or English, the language code or `forced_bos_token_id` is misconfigured.

In [ ]:
# Free training-time memory before reloading for inference.
del trainer
del model
import gc

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()

base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)
# ft_model = PeftModel.from_pretrained(base, BEST_DIR).to(DEVICE)
ft_model = base
ft_model.eval()

sample_en = [
    "How are you?",
    "The Lord is my shepherd, I shall not want.",
    "What is your name?",
    "I am going to the market.",
    "Good morning, my friend.",
    "She is reading a book.",
    "We need clean water to drink.",
    "The children are playing outside.",
    "Please come back tomorrow.",
    "Thank you very much.",
]

tokenizer.src_lang = SRC_LANG
forced_bos = tokenizer.convert_tokens_to_ids(TGT_LANG)
enc = tokenizer(sample_en, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
with torch.no_grad():
    out = ft_model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LENGTH, num_beams=4)
sample_tw = tokenizer.batch_decode(out, skip_special_tokens=True)
for en, tw in zip(sample_en, sample_tw):
    print(f"EN: {en}\nTW: {tw}\n")

HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8d333a098d19b4fd9a8b18f94170487ad3f821d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/commits/main "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/discussions?p=0 "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-distilled-600M/commits/refs%2Fpr%2F45 "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/refs%2Fpr%2F45/model.safetensors.index.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/refs%2Fpr%2F45/model.safetensors "HTTP/1.1 302 Found"
HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-distilled-600M/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-distilled-600M/f8

EN: How are you?
TW: Wo ho te dɛn?

EN: The Lord is my shepherd, I shall not want.
TW: Awurade ne me hwɛfo; hwee renhia me.

EN: What is your name?
TW: Wo din de dɛn?

EN: I am going to the market.
TW: Merekɔ gua so.

EN: Good morning, my friend.
TW: Anigyesɛm, me adamfo.

EN: She is reading a book.
TW: Ɔrekenkan nhoma bi.

EN: We need clean water to drink.
TW: Yehia nsu a ɛho tew sɛ yɛnom.

EN: The children are playing outside.
TW: Mmofra no redi agorɔ wɔ abɔnten.

EN: Please come back tomorrow.
TW: Mesrɛ wo san bra ɔkyena.

EN: Thank you very much.
TW: Meda mo ase paa.



## 8. Full evaluation — sacreBLEU + chrF++ + COMET-22

Three evaluations to underpin the paper:

1. **In-domain test** — 3k held out from `dataset/final.json` (fine-tuned model).
2. **Zero-shot baseline** — same 3k examples, scored with the un-fine-tuned NLLB-200. Gives the paper's "lift from fine-tuning" delta.
3. **FLORES-200 devtest** — the externally-citable low-resource MT benchmark (Twi side: `twi_Latn`).

Each evaluation is paired-bootstrap-tested against the zero-shot baseline so the paper can claim statistical significance.

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

test_src = splits["test"]["english"]
test_ref = splits["test"]["twi"]

eval_indomain_ft = evaluate_translation(
    ft_model, tokenizer, test_src, test_ref,
    device=DEVICE, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
    batch_size=8 if DEVICE == "mps" else 16,
    max_length=MAX_LENGTH, num_beams=4, compute_comet=True,
)
print(f"[in-domain  FT] BLEU={eval_indomain_ft['bleu']:.2f}  chrF++={eval_indomain_ft['chrf']:.2f}  COMET={eval_indomain_ft['comet']:.4f}")

In [ ]:
# Zero-shot baseline: same test set, un-fine-tuned base NLLB.
del ft_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()

zs_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)
zs_model.eval()

eval_indomain_zs = evaluate_translation(
    zs_model, tokenizer, test_src, test_ref,
    device=DEVICE, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
    batch_size=8 if DEVICE == "mps" else 16,
    max_length=MAX_LENGTH, num_beams=4, compute_comet=True,
)
print(f"[in-domain  ZS] BLEU={eval_indomain_zs['bleu']:.2f}  chrF++={eval_indomain_zs['chrf']:.2f}  COMET={eval_indomain_zs['comet']:.4f}")

In [ ]:
# FLORES-200 devtest (external benchmark). Use the `facebook/flores` dataset — each
# language is a separate config keyed by FLORES-200 code.
flores_eng = load_dataset("facebook/flores", "eng_Latn", split="devtest")
flores_twi = load_dataset("facebook/flores", "twi_Latn", split="devtest")
flores_src = [r["sentence"] for r in flores_eng]
flores_ref = [r["sentence"] for r in flores_twi]
print(f"FLORES-200 devtest: {len(flores_src)} aligned sentence pairs")

# Zero-shot first (model already loaded), then re-load FT for FLORES.
eval_flores_zs = evaluate_translation(
    zs_model, tokenizer, flores_src, flores_ref,
    device=DEVICE, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
    batch_size=8 if DEVICE == "mps" else 16,
    max_length=MAX_LENGTH, num_beams=4, compute_comet=True,
)
print(f"[FLORES    ZS] BLEU={eval_flores_zs['bleu']:.2f}  chrF++={eval_flores_zs['chrf']:.2f}  COMET={eval_flores_zs['comet']:.4f}")

del zs_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()

base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)
ft_model = PeftModel.from_pretrained(base, BEST_DIR).to(DEVICE)
ft_model.eval()

eval_flores_ft = evaluate_translation(
    ft_model, tokenizer, flores_src, flores_ref,
    device=DEVICE, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
    batch_size=8 if DEVICE == "mps" else 16,
    max_length=MAX_LENGTH, num_beams=4, compute_comet=True,
)
print(f"[FLORES    FT] BLEU={eval_flores_ft['bleu']:.2f}  chrF++={eval_flores_ft['chrf']:.2f}  COMET={eval_flores_ft['comet']:.4f}")

In [ ]:
# Paired bootstrap: FT vs ZS on each benchmark. Reviewers expect p < 0.05 for
# claims of significant improvement.
boot_indomain = paired_bootstrap(test_ref, eval_indomain_ft["predictions"], eval_indomain_zs["predictions"])
boot_flores = paired_bootstrap(flores_ref, eval_flores_ft["predictions"], eval_flores_zs["predictions"])
print("In-domain  chrF++  FT-ZS:", boot_indomain)
print("FLORES     chrF++  FT-ZS:", boot_flores)

## 9. Save artifacts for the paper

Writes everything a reviewer / replicator would want to see: metrics summary, full training config, predictions in TSV, and a markdown sample table for the appendix.

In [ ]:
def _dump_predictions(name, sources, refs, evals_dict):
    path = RESULTS_DIR / f"predictions_{name}.tsv"
    with open(path, "w", encoding="utf-8") as f:
        f.write("src\tref\thyp\tsent_chrf\n")
        for s, r, h, c in zip(sources, refs, evals_dict["predictions"], evals_dict["per_sentence_chrf"]):
            f.write(f"{s}\t{r}\t{h}\t{c:.4f}\n")
    print(f"  wrote {path}")


_dump_predictions("indomain_ft", test_src, test_ref, eval_indomain_ft)
_dump_predictions("indomain_zs", test_src, test_ref, eval_indomain_zs)
_dump_predictions("flores_ft", flores_src, flores_ref, eval_flores_ft)
_dump_predictions("flores_zs", flores_src, flores_ref, eval_flores_zs)


def _summary(d):
    # Drop bulky fields when writing the JSON summary.
    return {k: v for k, v in d.items() if k not in ("predictions", "references", "sources", "per_sentence_chrf", "per_sentence_comet")}


metrics_summary = {
    "model_id": MODEL_ID,
    "adapter_dir": BEST_DIR,
    "src_lang": SRC_LANG,
    "tgt_lang": TGT_LANG,
    "max_length": MAX_LENGTH,
    "num_beams": 4,
    "seed": SEED,
    "evaluations": {
        "indomain_ft": _summary(eval_indomain_ft),
        "indomain_zs": _summary(eval_indomain_zs),
        "flores_ft": _summary(eval_flores_ft),
        "flores_zs": _summary(eval_flores_zs),
    },
    "significance": {
        "indomain_ft_vs_zs_chrf": boot_indomain,
        "flores_ft_vs_zs_chrf": boot_flores,
    },
}
with open(RESULTS_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)
print("wrote", RESULTS_DIR / "metrics_summary.json")

training_config = {
    "model_id": MODEL_ID,
    "src_lang": SRC_LANG,
    "tgt_lang": TGT_LANG,
    "max_length": MAX_LENGTH,
    "device": DEVICE,
    "dtype": str(DTYPE),
    "lora": {"r": 16, "alpha": 32, "dropout": 0.1, "targets": ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]},
    "training_args": args.to_dict(),
    "split_sizes": {k: len(v) for k, v in splits.items()},
    "seed": SEED,
}
with open(RESULTS_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, default=str)
print("wrote", RESULTS_DIR / "training_config.json")

with open(RESULTS_DIR / "sample_translations.md", "w", encoding="utf-8") as f:
    f.write("# Sample translations (FLORES-200 devtest, fine-tuned model)\n\n")
    f.write(print_sample_table(flores_src, flores_ref, eval_flores_ft["predictions"], n=20))
print("wrote", RESULTS_DIR / "sample_translations.md")

## 10. (Optional) Push to Hugging Face Hub

Off by default. Set `PUSH_TO_HUB = True` and ensure `HF_TOKEN` is set in the environment.

In [ ]:
PUSH_TO_HUB = False
HUB_REPO_ID = "Joetib/nllb-200-twi-lora"

if PUSH_TO_HUB:
    if not HF_TOKEN:
        raise RuntimeError("Set HF_TOKEN before enabling PUSH_TO_HUB.")
    ft_model.push_to_hub(HUB_REPO_ID)
    tokenizer.push_to_hub(HUB_REPO_ID)
    print("Pushed to", HUB_REPO_ID)
else:
    print("PUSH_TO_HUB is False — skipping.")